Recommendation System

Data Description:

Unique ID of each anime.
Anime title.
Anime broadcast type, such as TV, OVA, etc.
anime genre.
The number of episodes of each anime.
The average rating for each anime compared to the number of users who gave ratings.


Number of community members for each anime.
Objective:
The objective of this assignment is to implement a recommendation system using cosine similarity on an anime dataset. 
Dataset:
Use the Anime Dataset which contains information about various anime, including their titles, genres,No.of episodes and user ratings etc.

Tasks:

Data Preprocessing:

Load the dataset into a suitable data structure (e.g., pandas DataFrame).
Handle missing values, if any.
Explore the dataset to understand its structure and attributes.

Feature Extraction:

Decide on the features that will be used for computing similarity (e.g., genres, user ratings).
Convert categorical features into numerical representations if necessary.
Normalize numerical features if required.

Recommendation System:

Design a function to recommend anime based on cosine similarity.
Given a target anime, recommend a list of similar anime based on cosine similarity scores.
Experiment with different threshold values for similarity scores to adjust the recommendation list size.
Analyze the performance of the recommendation system and identify areas of improvement.

Interview Questions:
1. Can you explain the difference between user-based and item-based collaborative filtering?
2. What is collaborative filtering, and how does it work?

Data Preprocessing:
Load the dataset into a suitable data structure (e.g., pandas DataFrame).
Handle missing values, if any.
Explore the dataset to understand its structure and attributes.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("anime.csv")

In [4]:
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [6]:
df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [8]:
df.describe()

,anime_id,rating,members
count,12017.000000,12017.000000,1.201700e+04
mean,13638.001165,6.478264,1.834888e+04
std,11231.076675,1.023857,5.537250e+04
min,1.000000,1.670000,1.200000e+01
25%,3391.000000,5.890000,2.250000e+02
50%,9959.000000,6.570000,1.552000e+03
75%,23729.000000,7.180000,9.588000e+03
max,34519.000000,10.000000,1.013917e+06


In [10]:
print("Rows and Columns:", df.shape)

Rows and Columns: (12017, 7)


1. Dataset loaded successfully.
2. Missing values were checked and removed.
3. Dataset information was examined.
4. Statistical summary was generated.
5. Dataset is ready for feature extraction.

Feature Extraction:

Decide on the features that will be used for computing similarity (e.g., genres, user ratings).
Convert categorical features into numerical representations if necessary.
Normalize numerical features if required.

In [11]:
from sklearn.preprocessing import MinMaxScaler

# Select required features
features = df[['genre', 'rating', 'episodes', 'members']].copy()

# Fill missing values
features['genre'] = features['genre'].fillna('')
features['rating'] = features['rating'].fillna(features['rating'].mean())
features['episodes'] = pd.to_numeric(features['episodes'], errors='coerce')
features['episodes'] = features['episodes'].fillna(features['episodes'].median())
features['members'] = features['members'].fillna(features['members'].median())

# Convert genre into dummy variables
genre_dummies = features['genre'].str.get_dummies(sep=',')

# Normalize numerical features
scaler = MinMaxScaler()
num_features = scaler.fit_transform(features[['rating', 'episodes', 'members']])

num_df = pd.DataFrame(
    num_features,
    columns=['rating', 'episodes', 'members']
)

# Final feature matrix
feature_matrix = pd.concat(
    [genre_dummies.reset_index(drop=True),
     num_df.reset_index(drop=True)],
    axis=1
)

print(feature_matrix.shape)
feature_matrix.head()

(12017, 85)


,Adventure,Cars,Comedy,Dementia,Demons,Drama,Ecchi,Fantasy,Game,Harem,...,Space,Sports,Super Power,Supernatural,Thriller,Vampire,Yaoi,rating,episodes,members
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0.924370,0.000000,0.197867
1,1,0,0,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0.911164,0.034673,0.782769
2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0.909964,0.027518,0.112683
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0.900360,0.012658,0.664323
4,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0.899160,0.027518,0.149180




1. Selected genre, rating, episodes and members as recommendation features.
2. Missing values were handled successfully.
3. Genre was converted into numerical format using one-hot encoding.
4. Numerical features were normalized using MinMaxScaler.
5. Final feature matrix is ready for cosine similarity.

Recommendation System:
Design a function to recommend anime based on cosine similarity.
Given a target anime, recommend a list of similar anime based on cosine similarity scores.
Experiment with different threshold values for similarity scores to adjust the recommendation list size.
Analyze the performance of the recommendation system and identify areas of improvement.


In [12]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity matrix
cosine_sim = cosine_similarity(feature_matrix)

# Reset index
df = df.reset_index(drop=True)

# Recommendation Function
def recommend_anime(anime_name, top_n=5, threshold=0.5):

    # Find anime index
    idx = df[df['name'] == anime_name].index

    if len(idx) == 0:
        return "Anime not found!"

    idx = idx[0]

    # Similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Apply threshold
    sim_scores = [x for x in sim_scores if x[1] >= threshold]

    # Sort scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Remove selected anime itself
    sim_scores = sim_scores[1:top_n+1]

    anime_indices = [i[0] for i in sim_scores]

    return df[['name', 'genre', 'rating']].iloc[anime_indices]

In [13]:
recommend_anime("Naruto", top_n=5, threshold=0.50)

,name,genre,rating
615,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",7.94
1472,Naruto: Shippuuden Movie 4 - The Lost Tower,"Action, Comedy, Martial Arts, Shounen, Super P...",7.53
1573,Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.50
486,Boruto: Naruto the Movie,"Action, Comedy, Martial Arts, Shounen, Super P...",8.03
1343,Naruto x UT,"Action, Comedy, Martial Arts, Shounen, Super P...",7.58


In [14]:
print("Threshold = 0.3")
print(recommend_anime("Naruto", threshold=0.3))

print("\nThreshold = 0.5")
print(recommend_anime("Naruto", threshold=0.5))

print("\nThreshold = 0.7")
print(recommend_anime("Naruto", threshold=0.7))

Threshold = 0.3
                                                   name  \
615                                  Naruto: Shippuuden   
1472        Naruto: Shippuuden Movie 4 - The Lost Tower   
1573  Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...   
486                            Boruto: Naruto the Movie   
1343                                        Naruto x UT   

                                                  genre  rating  
615   Action, Comedy, Martial Arts, Shounen, Super P...    7.94  
1472  Action, Comedy, Martial Arts, Shounen, Super P...    7.53  
1573  Action, Comedy, Martial Arts, Shounen, Super P...    7.50  
486   Action, Comedy, Martial Arts, Shounen, Super P...    8.03  
1343  Action, Comedy, Martial Arts, Shounen, Super P...    7.58  

Threshold = 0.5
                                                   name  \
615                                  Naruto: Shippuuden   
1472        Naruto: Shippuuden Movie 4 - The Lost Tower   
1573  Naruto: Shippuuden Movie 3 - Hi n



1. Cosine similarity was used to measure similarity between anime.
2. The recommendation function successfully returned similar anime.
3. Lower threshold values produced more recommendations.
4. Higher threshold values returned fewer but more relevant recommendations.
5. The recommendation quality depends on the selected features and dataset.

Interview Questions:
1. Can you explain the difference between user-based and item-based collaborative filtering?
2. What is collaborative filtering, and how does it work?

1

User-based collaborative filtering recommends items by finding users who have similar preferences. If two users have liked or rated similar anime in the past, the system recommends anime liked by one user to the other.

Item-based collaborative filtering recommends items by finding similarities between items instead of users. If a user likes a particular anime, the system recommends other anime that are similar to it based on ratings or features.

In simple words, user-based collaborative filtering focuses on similar users, while item-based collaborative filtering focuses on similar items. User-based filtering is more suitable for smaller datasets, whereas item-based filtering is generally faster and more efficient for large datasets.

2

Collaborative Filtering is a recommendation technique that suggests items to users based on the preferences and behavior of other users. It assumes that users with similar interests will like similar items.

Working:

Collect user-item interaction data (ratings, likes, purchases).
Calculate similarity between users or items.
Identify similar users or similar items.
Recommend the highest-rated or most similar items to the target user.

Example:
If User A and User B like similar anime, and User B likes One Piece but User A hasn't watched it, the system recommends One Piece to User A.